In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType, DecimalType, TimestampType, LongType
from delta.tables import *

StatementMeta(, 388adf83-1de8-4994-9bfa-d46cfbe004c3, 3, Finished, Available, Finished, False)

In [2]:
customer_silver = spark.sql("SELECT * FROM Olist_lh.dbo.customer_silver").alias("cs")
order_payments_silver = spark.sql("SELECT * FROM Olist_lh.dbo.order_payments_silver").alias("ops")
order_reviews_silver = spark.sql("SELECT * FROM Olist_lh.dbo.order_reviews_silver").alias("ors")
ordered_items_silver = spark.sql("SELECT * FROM Olist_lh.dbo.ordered_items_silver").alias("ois")
orders_silver = spark.sql("SELECT * FROM Olist_lh.dbo.orders_silver").alias("os")
products_silver = spark.sql("SELECT * FROM Olist_lh.dbo.products_silver").alias("ps")
sellers_silver = spark.sql("SELECT * FROM Olist_lh.dbo.sellers_silver").alias("ss")

StatementMeta(, 388adf83-1de8-4994-9bfa-d46cfbe004c3, 4, Finished, Available, Finished, False)

# **Buidling Gold Layer for DWH**

# **Building dim_customers**

In [3]:
dim_selected_customers = customer_silver.withColumn(
    "customer_key",
    F.abs(F.hash(F.col("customer_id")))
).select(
    "customer_key",
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state"
)

print(f"dim_selected_customers: {dim_selected_customers.count()} rows")
display(dim_selected_customers)
dim_selected_customers.schema

StatementMeta(, 388adf83-1de8-4994-9bfa-d46cfbe004c3, 5, Finished, Available, Finished, False)

dim_selected_customers: 1129 rows


SynapseWidget(Synapse.DataFrame, 9f320fd9-c588-44a6-a61d-d40b809bc38c)

StructType([StructField('customer_key', IntegerType(), False), StructField('customer_id', StringType(), True), StructField('customer_unique_id', StringType(), True), StructField('customer_zip_code_prefix', StringType(), True), StructField('customer_city', StringType(), True), StructField('customer_state', StringType(), True)])

In [30]:
# Define the customer gold schema
dim_customers_schema = StructType([
    StructField('customer_key', IntegerType(), False), 
    StructField('customer_id', StringType(), True), 
    StructField('customer_unique_id', StringType(), True), 
    StructField('customer_zip_code_prefix', StringType(), True), 
    StructField('customer_city', StringType(), True), 
    StructField('customer_state', StringType(), True)])

DeltaTable.createIfNotExists(spark).tableName("Dim_customers")\
           .addColumns(dim_customers_schema).execute()

StatementMeta(, d889256f-b268-486b-bb49-79756ba4999b, 32, Finished, Available, Finished, False)

# **creating a dim_customers using an UPSERT Logic**

In [32]:

dim_customers_table_path = "abfss://f9e66737-53ce-4d4c-bfdc-31f547b675c9@onelake.dfs.fabric.microsoft.com/de8ea8e4-7996-4472-932c-105896929b23/Tables/dbo/dim_customers"

dim_deltacustomers = DeltaTable.forPath(spark, dim_customers_table_path)

## perform the MERGE using pyspark (UPSERT)

dim_deltacustomers.alias("target").merge(
    dim_selected_customers.alias("source"),
    "target.customer_key = source.customer_key"
).whenMatchedUpdate(set = {
    "customer_id" : "source.customer_id",
    "customer_unique_id" : "source.customer_unique_id",
    "customer_zip_code_prefix" : "source.customer_zip_code_prefix",
    "customer_city" : "source.customer_city",
    "customer_state" : "source.customer_state"

}).whenNotMatchedInsert(values = {
    "customer_key" : "source.customer_key",
    "customer_id" : "source.customer_id",
    "customer_unique_id" : "source.customer_unique_id",
    "customer_zip_code_prefix" : "source.customer_zip_code_prefix",
    "customer_city" : "source.customer_city",
    "customer_state" : "source.customer_state"

}).execute()

StatementMeta(, d889256f-b268-486b-bb49-79756ba4999b, 34, Finished, Available, Finished, False)

In [ ]:
# get the history of the delta table 
history_df = dim_deltacustomers.history(1) # get the latest operation

# extract metrics from the history df
operation_metrics = history.df.select("operationMetrics").collect()[0][0]

# extract specific metrics
rows_inserted = operation_metrics.get('numTargetRowsInserted', 0)
rows_updated = operation_metrics.get('numTargetRowsUpdated', 0)
rows_deleted = operation_metrics.get('numTargetRowsDeleted', 0)
rows_affected = int(rows_inserted) + int(rows_updated) + int(rows_deleted)

print('total rows of table: ',dim_deltacustomers.toDF().count())
print("Merge Metrics:")
print(f"rows inserted: {rows_inserted}")
print(f"rows updated: {rows_updated}")
print(f"rows deleted: {rows_deleted}")
print(f"total rows_affected: {rows_affected}")



# **Building dim_products**

In [4]:
# select product attributes needed
# creat a product_key
dim_selected_products = products_silver.withColumn(
    "product_key",
    F.abs(F.hash(F.col("product_id")))
).select(
    "product_key",
    "product_id",
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_weight_gram",
    "product_length_cm",
    "product_width_cm",
    "product_category_name"
)

print(f"dim_selected_products: {dim_selected_products.count()} rows")
display(dim_selected_products)
dim_selected_products.schema

StatementMeta(, 388adf83-1de8-4994-9bfa-d46cfbe004c3, 6, Finished, Available, Finished, False)

dim_selected_products: 859 rows


SynapseWidget(Synapse.DataFrame, 879566cd-7498-45f6-a6f8-7aa943d76866)

StructType([StructField('product_key', IntegerType(), False), StructField('product_id', StringType(), True), StructField('product_name_length', DecimalType(10,2), True), StructField('product_description_length', DecimalType(10,2), True), StructField('product_photos_qty', DecimalType(10,2), True), StructField('product_weight_gram', DecimalType(10,2), True), StructField('product_length_cm', DecimalType(10,2), True), StructField('product_width_cm', DecimalType(10,2), True), StructField('product_category_name', StringType(), True)])

In [12]:
dim_products_schema = StructType([
    StructField('product_key', IntegerType(), False), 
    StructField('product_id', StringType(), True), 
    StructField('product_name_length', DecimalType(10,2), True), 
    StructField('product_description_length', DecimalType(10,2), True), 
    StructField('product_photos_qty', DecimalType(10,2), True), 
    StructField('product_weight_gram', DecimalType(10,2), True), 
    StructField('product_length_cm', DecimalType(10,2), True), 
    StructField('product_width_cm', DecimalType(10,2), True), 
    StructField('product_category_name', StringType(), True)])

DeltaTable.createIfNotExists(spark).tableName("Dim_products")\
           .addColumns(dim_products_schema).execute()

StatementMeta(, d889256f-b268-486b-bb49-79756ba4999b, 14, Finished, Available, Finished, False)

# **creating dim_products using UPSERT Logic**

In [ ]:
dim_customers_table_path = "abfss://f9e66737-53ce-4d4c-bfdc-31f547b675c9@onelake.dfs.fabric.microsoft.com/de8ea8e4-7996-4472-932c-105896929b23/Tables/dbo/dim_products"


dim_deltaproducts = DeltaTable.forPath(spark, dim_customers_table_path)

## perform the MERGE using pyspark (UPSERT)

dim_deltaproducts.alias("target").merge(
    dim_selected_products.alias("source"),
    "target.product_key = source.product_key"
).whenMatchedUpdate(set = {
    "product_id" : "sorce.product_id",
    "product_name_length" : "sorce.product_name_length",
    "product_description_length" : "sorce.product_description_length",
    "product_photos_qty" : "sorce.product_photos_qty",
    "product_weight_gram" : "sorce.product_weight_gram",
    "product_length_cm" : "sorce.product_length_cm",
    "product_width_cm" : "sorce.product_width_cm",
    "product_category_name" : "sorce.product_category_name"

}).whenNotMatchedInsert(values = {
    "product_key" : "source.product_key",
    "product_id" : "sorce.product_id",
    "product_name_length" : "sorce.product_name_length",
    "product_description_length" : "sorce.product_description_length",
    "product_photos_qty" : "sorce.product_photos_qty",
    "product_weight_gram" : "sorce.product_weight_gram",
    "product_length_cm" : "sorce.product_length_cm",
    "product_width_cm" : "sorce.product_width_cm",
    "product_category_name" : "sorce.product_category_name"

}).execute()

In [ ]:
# get the history of the delta table 
history_df = dim_deltaproducts.history(1) # get the latest operation

# extract metrics from the history df
operation_metrics = history.df.select("operationMetrics").collect()[0][0]

# extract specific metrics
rows_inserted = operation_metrics.get('numTargetRowsInserted', 0)
rows_updated = operation_metrics.get('numTargetRowsUpdated', 0)
rows_deleted = operation_metrics.get('numTargetRowsDeleted', 0)
rows_affected = int(rows_inserted) + int(rows_updated) + int(rows_deleted)

print('total rows of table: ',dim_deltaproducts.toDF().count())
print("Merge Metrics:")
print(f"rows inserted: {rows_inserted}")
print(f"rows updated: {rows_updated}")
print(f"rows deleted: {rows_deleted}")
print(f"total rows_affected: {rows_affected}")



# **Building dim_sellers**

In [5]:
# select seller attributes
# create a seller key

dim_selected_sellers = sellers_silver.withColumn(
    "seller_key",
    F.abs(F.hash(F.col("seller_id")))
).select(
    "seller_key",
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state"
)

print(f"dim_selected_sellers: {dim_selected_sellers.count()} rows")
dim_selected_sellers.schema

StatementMeta(, 388adf83-1de8-4994-9bfa-d46cfbe004c3, 7, Finished, Available, Finished, False)

dim_selected_sellers: 296 rows


StructType([StructField('seller_key', IntegerType(), False), StructField('seller_id', StringType(), True), StructField('seller_zip_code_prefix', StringType(), True), StructField('seller_city', StringType(), True), StructField('seller_state', StringType(), True)])

In [14]:
dim_sellers_schema = StructType([
    StructField('seller_key', IntegerType(), False), 
    StructField('seller_id', StringType(), True), 
    StructField('seller_zip_code_prefix', StringType(), True), 
    StructField('seller_city', StringType(), True), 
    StructField('seller_state', StringType(), True)])

DeltaTable.createIfNotExists(spark).tableName("Dim_sellers")\
           .addColumns(dim_sellers_schema).execute()

StatementMeta(, d889256f-b268-486b-bb49-79756ba4999b, 16, Finished, Available, Finished, False)

# **creating a dim_sellers using an UPSERT Logic**

In [ ]:
dim_sellers_table_path = "abfss://f9e66737-53ce-4d4c-bfdc-31f547b675c9@onelake.dfs.fabric.microsoft.com/de8ea8e4-7996-4472-932c-105896929b23/Tables/dbo/dim_sellers"


dim_deltasellers = DeltaTable.forPath(spark, dim_sellers_table_path)

## perform the MERGE using pyspark (UPSERT)

dim_deltasellers.alias("target").merge(
    dim_selected_sellers.alias("source"),
    "target.seller_key = source.seller_key"
).whenMatchedUpdate(set = {
    "seller_id" : "source.seller_id",
    "seller_zip_code_prefix" : "source.seller_zip_code_prefix",
    "seller_city" : "source.seller_city",
    "seller_state" : "source.seller_state"

}).whenNotMatchedInsert(values = {
    "seller_key" : "source.seller_key",
    "seller_id" : "source.seller_id",
    "seller_zip_code_prefix" : "source.seller_zip_code_prefix",
    "seller_city" : "source.seller_city",
    "seller_state" : "source.seller_state"

}).execute()


In [ ]:
# get the history of the delta table 
history_df = dim_deltasellers.history(1) # get the latest operation

# extract metrics from the history df
operation_metrics = history.df.select("operationMetrics").collect()[0][0]

# extract specific metrics
rows_inserted = operation_metrics.get('numTargetRowsInserted', 0)
rows_updated = operation_metrics.get('numTargetRowsUpdated', 0)
rows_deleted = operation_metrics.get('numTargetRowsDeleted', 0)
rows_affected = int(rows_inserted) + int(rows_updated) + int(rows_deleted)

print('total rows of table: ',dim_deltasellers.toDF().count())
print("Merge Metrics:")
print(f"rows inserted: {rows_inserted}")
print(f"rows updated: {rows_updated}")
print(f"rows deleted: {rows_deleted}")
print(f"total rows_affected: {rows_affected}")



# **Building the fact order lines**
# 1. aggregate ordered items to get quantity per product per order
# 2. group by order_id, product_id, seller_id, shipping_limit_date
# 3. count the number of rows = quantity
# 4. use max(order_item_id) as line_number

In [6]:
# creating initial sales order lines

order_items_agg = ordered_items_silver \
     .groupBy("order_id", "product_id", "seller_id", "shipping_limit_date") \
     .agg(
        F.count("order_item_id").alias("item_quantity"),
        F.round(F.sum("price"), 2).alias("line_revenue"),
        F.round(F.avg("price"), 2).alias("unit_price"),
        F.round(F.sum("freight_value"), 2).alias("freight_value"),
        F.min("price").alias("min_unit_price"),
        F.max("price").alias("max_unit_price")
     ) \
     .withColumn(
        "price_variance_flag",
        F.when(F.col("min_unit_price") != F.col("max_unit_price"), "Y").otherwise("N")
     )

# Join with dimension df
selected_fact_order_lines = order_items_agg \
     .join(
        dim_selected_products.select(
            "product_key", "product_id"),
            on="product_id", how = "left"
        ) \
        .join(
            dim_selected_sellers.select("seller_key", "seller_id"),
            on="seller_id", how="left"
        ) \
        .withColumn(
            "order_line_key",
            F.abs(F.hash(F.concat(
                F.col("order_id"),
                F.col("product_id"),
                F.col("seller_id"),
                F.col("shipping_limit_date").cast("string")
            )))
        ) \
        .select(
            "order_line_key", # creating a unique key that identifies each row
            "order_id",
            "product_key",# replacing product_id with product_key
            "seller_key", # replacing seller id with seller_key
            "shipping_limit_date",
            "item_quantity",
            "line_revenue",
            "unit_price",
            "freight_value",
            "min_unit_price",
            "max_unit_price",
            "price_variance_flag"
        )

print(f"selected_fact_order_lines: {selected_fact_order_lines.count()} rows")
display(selected_fact_order_lines)

StatementMeta(, 388adf83-1de8-4994-9bfa-d46cfbe004c3, 8, Finished, Available, Finished, False)

selected_fact_order_lines: 1156 rows


SynapseWidget(Synapse.DataFrame, a0d5e850-56d4-42e4-8203-cdaa97a3cca3)

# **Creating a fact orders table**

# **Aggregate the following tables - order_payments, order_reviews, and fact_order_lines**

In [7]:
payments_agg = order_payments_silver \
     .groupBy(
        "order_id") \
        .agg(
           F.first("payment_type").alias("payment_type"),
           F.round(F.sum("payment_value"), 2).alias("total_payment_value"),
           F.max("payment_installments").alias("payment_installments")
        )


# aggregate reviews per order - one review per order
reviews_agg = order_reviews_silver \
    .groupBy("order_id") \
    .agg(
        F.avg("review_score").alias("review_score")
    )

# aggregate order lines for order level metrics
order_lines_agg = selected_fact_order_lines \
      .groupBy("order_id") \
      .agg(
        F.countDistinct("order_line_key").alias("total_line_items"),
        F.sum("item_quantity").alias("total_quantity"),
        F.round(F.sum("line_revenue"), 2).alias("total_revenue"),
        F.round(F.sum("freight_value"), 2).alias("total_freight"),
        F.round(F.avg("unit_price"), 2).alias("average_item_value")
      )

StatementMeta(, 388adf83-1de8-4994-9bfa-d46cfbe004c3, 9, Finished, Available, Finished, False)

In [8]:
# Aggregating payments and reviews for fact orders


selected_fact_orders = orders_silver \
    .join(payments_agg, on="order_id", how = 'left') \
    .join(reviews_agg, on="order_id", how = 'left') \
    .join(order_lines_agg, on="order_id", how = "left") \
    .join(
        dim_selected_customers.select("customer_key", "customer_id"),
        on = "customer_id", how = "left"
    ) \
    .withColumn( 
        # creating days from order placed to approval
    "days_to_approve",
    F.datediff(
        F.col("order_approved_at"), F.col("order_purchase_timestamp")
    ) 
    ) \
    .withColumn(
        # days from approval to shipped
        "days_to_ship",
        F.date_diff(F.col("order_delivered_carrier_date"), F.col("order_approved_at"))
    ) \
    .withColumn(
        "days_estimated",
        F.datediff(F.col("order_estimated_delivery_date"), F.col("order_purchase_timestamp"))
    ) \
    .withColumn(
        "days_to_deliver",
        F.datediff(F.col("order_delivered_customer_date"), F.col("order_purchase_timestamp"))
    ) \
    .withColumn(
        # delivery variance
        "delivery_variance_days",
        F.datediff(
            F.col("order_delivered_customer_date"), F.col("order_estimated_delivery_date")
        )
    ) \
    .withColumn(
        # delivery status flag
        "delivery_status_flag",
        F.when(F.col("order_delivered_customer_date") <= F.col("order_estimated_delivery_date"),
        "On Time").otherwise("Late")
    ) \
    .withColumn(
        # extract order month
        "order_month",
        F.date_format(F.col("order_purchase_timestamp"), "yyyy-MM")
    ) \
    .withColumn(
        # extract order year
        "order_year",
        F.year(F.col("order_purchase_timestamp"))
    ) \
    .withColumn(
        "order_key",
        F.abs(F.hash(F.col("order_id")))
    ) \
    .select(
        "order_key",
        "order_id",
        "customer_key", # replace customer_id with customer_key
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "payment_type",
        "payment_installments",
        "total_payment_value",
        "total_line_items",
        "total_quantity",
        "total_revenue",
        "total_freight",
        "average_item_value",
        "days_to_approve",
        "days_to_ship",
        "days_to_deliver",
        "days_estimated",
        "delivery_variance_days",
        "delivery_status_flag",
        "review_score",
        "order_month",
        "order_year"
    )


selected_fact_orders.schema

StatementMeta(, 388adf83-1de8-4994-9bfa-d46cfbe004c3, 10, Finished, Available, Finished, False)

StructType([StructField('order_key', IntegerType(), False), StructField('order_id', StringType(), True), StructField('customer_key', IntegerType(), True), StructField('order_status', StringType(), True), StructField('order_purchase_timestamp', TimestampType(), True), StructField('order_approved_at', TimestampType(), True), StructField('order_delivered_carrier_date', TimestampType(), True), StructField('order_delivered_customer_date', TimestampType(), True), StructField('order_estimated_delivery_date', TimestampType(), True), StructField('payment_type', StringType(), True), StructField('payment_installments', IntegerType(), True), StructField('total_payment_value', DecimalType(21,2), True), StructField('total_line_items', LongType(), True), StructField('total_quantity', LongType(), True), StructField('total_revenue', DecimalType(32,2), True), StructField('total_freight', DecimalType(32,2), True), StructField('average_item_value', DecimalType(12,2), True), StructField('days_to_approve', 

In [24]:
fact_orders_schema = StructType([
    StructField('order_key', IntegerType(), False), 
    StructField('order_id', StringType(), True), 
    StructField('customer_key', IntegerType(), True), 
    StructField('order_status', StringType(), True), 
    StructField('order_purchase_timestamp', TimestampType(), True), 
    StructField('order_approved_at', TimestampType(), True), 
    StructField('order_delivered_carrier_date', TimestampType(), True), 
    StructField('order_delivered_customer_date', TimestampType(), True), 
    StructField('order_estimated_delivery_date', TimestampType(), True), 
    StructField('payment_type', StringType(), True), 
    StructField('payment_installments', IntegerType(), True), 
    StructField('total_payment_value', DecimalType(21,2), True), 
    StructField('total_line_items', LongType(), True), 
    StructField('total_quantity', LongType(), True), 
    StructField('total_revenue', DecimalType(32,2), True), 
    StructField('total_freight', DecimalType(32,2), True), 
    StructField('average_item_value', DecimalType(12,2), True), 
    StructField('days_to_approve', IntegerType(), True), 
    StructField('days_to_ship', IntegerType(), True), 
    StructField('days_to_deliver', IntegerType(), True), 
    StructField('days_estimated', IntegerType(), True), 
    StructField('delivery_variance_days', IntegerType(), True), 
    StructField('delivery_status_flag', StringType(), False), 
    StructField('review_score', DoubleType(), True), 
    StructField('order_month', StringType(), True), 
    StructField('order_year', IntegerType(), True)])


DeltaTable.createIfNotExists(spark).tableName("fact_orders")\
           .addColumns(fact_orders_schema).execute()

StatementMeta(, d889256f-b268-486b-bb49-79756ba4999b, 26, Finished, Available, Finished, False)

# **creating a fact_orders using an UPSERT Logic**

In [ ]:
fact_orders_table_path = "abfss://f9e66737-53ce-4d4c-bfdc-31f547b675c9@onelake.dfs.fabric.microsoft.com/de8ea8e4-7996-4472-932c-105896929b23/Tables/dbo/fact_orders"

fact_deltaorders = DeltaTable.forPath(spark, fact_orders_table_path)

## perform the MERGE using pyspark (UPSERT)

fact_deltaorders.alias("target").merge(
    selected_fact_orders.alias("source"),
    "target.order_key = source.order_key"
).whenMatchedUpdate(set = {
        "order_id" : "source.order_id" ,
        "customer_key" : "source.customer_key",
        "order_status" : "source.order_status",
        "order_purchase_timestamp" : "source.order_purchase_timestamp",
        "order_approved_at" : "source.order_approved_at",
        "order_delivered_carrier_date" : "source.order_delivered_carrier_date",
        "order_delivered_customer_date" : "vorder_delivered_customer_date",
        "order_estimated_delivery_date" : "source.order_estimated_delivery_date",
        "payment_type" : "source.payment_type",
        "payment_installments" : "source.payment_installments",
        "total_payment_value" : "source.total_payment_value",
        "total_line_items" : "source.total_line_items",
        "total_quantity" : "source.total_quantity",
        "total_revenue" : "source.total_revenue",
        "total_freight" : "source.total_freight",
        "average_item_value" : "source.average_item_value",
        "days_to_approve" : "source.days_to_approve",
        "days_to_ship" : "source.days_to_ship",
        "days_to_deliver" : "source.days_to_deliver",
        "days_estimated" : "source.days_estimated",
        "delivery_variance_days" : "source.delivery_variance_days",
        "delivery_status_flag" : "source.delivery_status_flag",
        "review_score" : "source.review_score",
        "order_month" : "source.order_month",
        "order_year" : "source.order_year"

}).whenNotMatchedInsert(values = {
        "order_key" : "source.order_key",
        "order_id" : "source.order_id" ,
        "customer_key" : "source.customer_key",
        "order_status" : "source.order_status",
        "order_purchase_timestamp" : "source.order_purchase_timestamp",
        "order_approved_at" : "source.order_approved_at",
        "order_delivered_carrier_date" : "source.order_delivered_carrier_date",
        "order_delivered_customer_date" : "vorder_delivered_customer_date",
        "order_estimated_delivery_date" : "source.order_estimated_delivery_date",
        "payment_type" : "source.payment_type",
        "payment_installments" : "source.payment_installments",
        "total_payment_value" : "source.total_payment_value",
        "total_line_items" : "source.total_line_items",
        "total_quantity" : "source.total_quantity",
        "total_revenue" : "source.total_revenue",
        "total_freight" : "source.total_freight",
        "average_item_value" : "source.average_item_value",
        "days_to_approve" : "source.days_to_approve",
        "days_to_ship" : "source.days_to_ship",
        "days_to_deliver" : "source.days_to_deliver",
        "days_estimated" : "source.days_estimated",
        "delivery_variance_days" : "source.delivery_variance_days",
        "delivery_status_flag" : "source.delivery_status_flag",
        "review_score" : "source.review_score",
        "order_month" : "source.order_month",
        "order_year" : "source.order_year"

}).execute()




# **Creating Order Line schema**

In [25]:
selected_fact_order_lines = selected_fact_order_lines \
     .join(
        selected_fact_orders.select("order_id", "order_key"),
        on="order_id", how="left"
     ) \
     .select(
        "order_line_key",
        "order_key",
        "order_id",
        "product_key",
        "seller_key",
        "item_quantity",
        "line_revenue",
        "unit_price",
        "freight_value",
        "shipping_limit_date",

     )

display(selected_fact_order_lines)
selected.fact_order_lines.schema

StatementMeta(, d889256f-b268-486b-bb49-79756ba4999b, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, edb65279-0c35-4c14-b15e-595ba0bd9260)

StructType([StructField('order_line_key', IntegerType(), False), StructField('order_key', IntegerType(), True), StructField('order_id', StringType(), True), StructField('product_key', IntegerType(), True), StructField('seller_key', IntegerType(), True), StructField('item_quantity', LongType(), False), StructField('line_revenue', DecimalType(21,2), True), StructField('unit_price', DecimalType(11,2), True), StructField('freight_value', DecimalType(21,2), True), StructField('shipping_limit_date', TimestampType(), True)])

In [26]:
fact_order_lines_schema = StructType([
    StructField('order_line_key', IntegerType(), False), 
    StructField('order_key', IntegerType(), True), 
    StructField('order_id', StringType(), True), 
    StructField('product_key', IntegerType(), True), 
    StructField('seller_key', IntegerType(), True), 
    StructField('item_quantity', LongType(), False), 
    StructField('line_revenue', DecimalType(21,2), True), 
    StructField('unit_price', DecimalType(11,2), True), 
    StructField('freight_value', DecimalType(21,2), True), 
    StructField('shipping_limit_date', TimestampType(), True)])


DeltaTable.createIfNotExists(spark).tableName("fact_order_lines")\
           .addColumns(fact_order_lines_schema).execute()

StatementMeta(, d889256f-b268-486b-bb49-79756ba4999b, 28, Finished, Available, Finished, False)

In [ ]:
fact_order_lines_table_path = "abfss://f9e66737-53ce-4d4c-bfdc-31f547b675c9@onelake.dfs.fabric.microsoft.com/de8ea8e4-7996-4472-932c-105896929b23/Tables/dbo/fact_order_lines"

fact_deltaorderlines = DeltaTable.forPath(spark, fact_order_lines_table_path)

# Perform the MERGE using UPSERT function:


fact_deltaorderlines.alias("target").merge(
    fact_deltaorderlines.alias("source"),
    "target.order_line_key = source.order_line_key"
).whenMatchedUpdate(set = {
        "order_key" : "source.order_key",
        "order_id" : "source.order_id",
        "product_key" : "source.product_key",
        "seller_key" : "source.seller_key",
        "item_quantity" : "source.item_quantity",
        "line_revenue" : "source.line_revenue",
        "unit_price" : "source.unit_price",
        "freight_value" : "source.freight_value",
        "shipping_limit_date" : "source.shipping_limit_date"

}).whenNotMatchedInsert(values = {
        "order_line_key" : "source.order_line_key",
        "order_key" : "source.order_key",
        "order_id" : "source.order_id",
        "product_key" : "source.product_key",
        "seller_key" : "source.seller_key",
        "item_quantity" : "source.item_quantity",
        "line_revenue" : "source.line_revenue",
        "unit_price" : "source.unit_price",
        "freight_value" : "source.freight_value",
        "shipping_limit_date" : "source.shipping_limit_date"

}).execute()
